In [1]:
###Cell 1: install dependencies

! pip install openai mlflow[kubernetes]

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cpu-ubi9/simple/


In [2]:
###Cell 2: hardcoded prompts

input_text = """
What's the difference between RHEL and CentOS?
"""
system_instructions = """
You are a helpful AI assistant.
You are designed to answer questions in a concise and professional manner.
"""

In [3]:
###Cell 3: environment settings

import os

BASE_URL = "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1"
model_name = "llama-32-3b-instruct"

MLFLOW_TRACKING_URL = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
MLFLOW_EXPERIMENT = "simple-agent"

os.environ["MLFLOW_TRACKING_AUTH"]="kubernetes-namespaced"

In [4]:
###Cell 4: start tracing

import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URL)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tracing.disable_notebook_display()
mlflow.openai.autolog()

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
###Cell 5: simplistic agent

from openai import OpenAI

def agent (key: str, system: str, query: str) -> str:
    client = OpenAI(
        base_url=BASE_URL,
        api_key=key
    )
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": query}
        ],
        max_tokens=256,
        temperature=0.7
    )

    return response.choices[0].message.content

In [6]:
###Cell 6: run the agent

# don't need this because the inferenceservice is setup with token auth disabled
#try:
#    with open("/var/run/secrets/kubernetes.io/serviceaccount/token", "r") as f:
#        AUTH_TOKEN = f.read().strip()
#except FileNotFoundError:
#    # Fallback if running outside a workbench or using a manually generated cluster token
#    AUTH_TOKEN = OCP_TOKEN

try:
    response = agent("no token", system_instructions, input_text)    
    print(response)
except Exception as e:
    print(f"An error occurred: {e}")

Red Hat Enterprise Linux (RHEL) and CentOS are both Linux distributions, but they have distinct differences:

1. **Licensing**: RHEL is a proprietary Linux distribution from Red Hat, requiring a license fee. CentOS, on the other hand, is a community-supported, open-source distribution based on RHEL.
2. **Support**: RHEL receives regular security updates and technical support from Red Hat, while CentOS relies on the community for support and updates.
3. **Source Code**: RHEL is built from Red Hat's internal source code, while CentOS is built from RHEL's source code, which is made available to the community.
4. **Upstream Support**: RHEL is directly supported by Red Hat, while CentOS is supported by the community, with some support from Red Hat.
5. **Upgrade Policy**: RHEL follows a more aggressive upgrade policy, with new releases typically released every two years. CentOS, however, tends to follow a more conservative upgrade policy, with new releases typically released every five years